<a href="https://colab.research.google.com/github/sergiocalazans/big-data-pyspark/blob/main/PairRDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#importar bibliotecas
import pyspark
from pyspark import SparkFiles

#criar spark session
sc = pyspark.SparkContext('local')

In [ ]:
#creating a PairRDD from a list

list_tuples = [(1,"Banana"), (2, "Melão"), (3, "Melancia")]

pair_rdd = sc.parallelize(list_tuples)
pair_rdd.collect()

[(1, 'Banana'), (2, 'Melão'), (3, 'Melancia')]

In [ ]:
#Criar a partir de um RDD
rdd = sc.parallelize(["12345, Abacate","12346, Banana", "12347, Maça"])
pair_rdd = rdd.map(lambda x: (x.split(",")[0],x.split(",")[1]))

pair_rdd.take(5)

[('12345', ' Abacate'), ('12346', ' Banana'), ('12347', ' Maça')]

In [ ]:
#criar função extra
def criar_rdd(rdd):
  y = rdd.split(",")
  return (y[0],y[1])

rdd = sc.parallelize(["12345, Abacate","12346, Banana", "12347, Maça"])
pair_rdd = rdd.map(criar_rdd)

pair_rdd.take(5)

[('12345', ' Abacate'), ('12346', ' Banana'), ('12347', ' Maça')]

In [ ]:
#Exercício 1

#copiar url
url = 'https://raw.githubusercontent.com/joegotz/bigDataAnalytics/main/euro2024_players.csv'

#adicionar arquivo ao contexto
sc.addFile(url)

In [ ]:
########################## #Exercício 1
#1. criar rdd a partir do arquivo baixado
#2. Transformar o rdd em pairRdd -- (name, country)
#3. Aplicar filter para deixar apenas dados da Alemanha
#4. Capitalize only the values of Conutry
#5. Realizar ordenação baseado na chave (nome do jogador)
#6. Mostrar resultado

In [ ]:
#Exercício 1
#1. criar rdd a partir do arquivo baixado
rdd = sc.textFile( SparkFiles.get('euro2024_players.csv'))
rdd.take(5)

#2. Transformar o rdd em pairRdd -- (name, country)
pairRdd = rdd.map(lambda x: (x.split(",")[0], x.split(",")[-1]))

#3. Aplicar filter para deixar apenas dados da Alemanha
pairRdd = pairRdd.filter(lambda x: x[1]=='Germany')

#4. Capitalize only the values of Conutry
pairRdd = pairRdd.mapValues(lambda x: x.capitalize())

#5. Realizar ordenação baseado na chave (nome do jogador)
pairRdd = pairRdd.sortByKey(ascending=True)

#6. Mostrar resultado
pairRdd.take(5)

[('Aleksandar Pavlovic', 'Germany'),
 ('Antonio Rüdiger', 'Germany'),
 ('Benjamin Henrichs', 'Germany'),
 ('Chris Führich', 'Germany'),
 ('David Raum', 'Germany')]

In [ ]:
#### Challenge
#1. Load the Euro2024.csv dataset as a conventional RDD and remove the header
#2. Convert it into a PairRDD with (Name, (Age, Goals,Country)) format
#3. Obtain all entries in which the Country is equal to GERMANY and AGE <25
#4. Capitalize only the Country, but maintaining the format (k, (v1,v2,v3))
#5. Sort the results according to the Age
#6. Average age for country
#7. Average goals for country

In [ ]:
#1. Load the Euro2024.csv dataset as a conventional RDD and remove the header
rdd = sc.textFile( SparkFiles.get('euro2024_players.csv'))
header = rdd.first()
rdd = rdd.filter(lambda x: x != header)
rdd.take(5)


['Marc-André ter Stegen,Goalkeeper,32,FC Barcelona,187,right,40,0,28000000,Germany',
 'Manuel Neuer,Goalkeeper,38,Bayern Munich,193,right,119,0,4000000,Germany',
 'Oliver Baumann,Goalkeeper,34,TSG 1899 Hoffenheim,187,right,0,0,3000000,Germany',
 'Nico Schlotterbeck,Centre-Back,24,Borussia Dortmund,191,left,12,0,40000000,Germany',
 'Jonathan Tah,Centre-Back,28,Bayer 04 Leverkusen,195,right,25,0,30000000,Germany']

In [ ]:
#2. Convert it into a PairRDD with (Name, (Age, Goals,Country)) format
pairRdd = rdd.map(lambda x: (x.split(",")[0], (x.split(",")[2], x.split(",")[-3], x.split(",")[-1])))
pairRdd.take(5)

[('Marc-André ter Stegen', ('32', '0', 'Germany')),
 ('Manuel Neuer', ('38', '0', 'Germany')),
 ('Oliver Baumann', ('34', '0', 'Germany')),
 ('Nico Schlotterbeck', ('24', '0', 'Germany')),
 ('Jonathan Tah', ('28', '0', 'Germany'))]

In [ ]:
#3. Obtain all entries in which the Country is equal to GERMANY and AGE <25
pairRdd_germany = pairRdd.filter(lambda x: x[1][2]=='Germany' and int(x[1][0])<25)
pairRdd_germany.take(5)

[('Nico Schlotterbeck', ('24', '0', 'Germany')),
 ('Aleksandar Pavlovic', ('20', '0', 'Germany')),
 ('Florian Wirtz', ('21', '1', 'Germany')),
 ('Jamal Musiala', ('21', '2', 'Germany')),
 ('Kai Havertz', ('24', '16', 'Germany'))]

In [ ]:
#4. Capitalize only the Country, but maintaining the format (k, (v1,v2,v3))
pairRdd = pairRdd.map(lambda x: (x[0], (x[1][0], x[1][1], x[1][2].capitalize())))
pairRdd.take(5)

[('Marc-André ter Stegen', ('32', '0', 'Germany')),
 ('Manuel Neuer', ('38', '0', 'Germany')),
 ('Oliver Baumann', ('34', '0', 'Germany')),
 ('Nico Schlotterbeck', ('24', '0', 'Germany')),
 ('Jonathan Tah', ('28', '0', 'Germany'))]

In [ ]:
#5. Sort the results according to the Age
pairRdd_age = pairRdd.sortBy(lambda x: int(x[1][0]))
pairRdd_age.take(5)

[('Lamine Yamal', ('16', '2', 'Spain')),
 ('Warren Zaïre-Emery', ('18', '1', 'France')),
 ('Leo Sauer', ('18', '0', 'Slovakia')),
 ('Semih Kılıçsoy', ('18', '0', 'Turkiye')),
 ('Gabriel Sigua', ('18', '0', 'Georgia'))]

In [ ]:
#6. Average age for country
pairRdd_age_country = pairRdd.map(lambda x: (x[1][2], (int(x[1][0]), 1)))
pairRdd_age_country = pairRdd_age_country.reduceByKey(lambda x,y: (x[0]+y[0], x[1]+y[1])).mapValues(lambda x: x[0]/x[1])
pairRdd_age_country.collect()

[('Germany', 28.115384615384617),
 ('Scotland', 28.23076923076923),
 ('Hungary', 27.423076923076923),
 ('Switzerland', 27.692307692307693),
 ('Spain', 26.96153846153846),
 ('Croatia', 27.692307692307693),
 ('Albania', 27.307692307692307),
 ('Italy', 26.5),
 ('Slovenia', 27.346153846153847),
 ('England', 26.076923076923077),
 ('Denmark', 27.692307692307693),
 ('Serbia', 27.384615384615383),
 ('Poland', 27.555555555555557),
 ('Netherlands', 26.53846153846154),
 ('Austria', 26.807692307692307),
 ('France', 26.88),
 ('Belgium', 26.88),
 ('Slovakia', 27.26923076923077),
 ('Romania', 27.0),
 ('Ukraine', 26.346153846153847),
 ('Turkiye', 25.807692307692307),
 ('Georgia', 27.153846153846153),
 ('Portugal', 26.96153846153846),
 ('Czech republic', 25.307692307692307)]

In [ ]:
#7. Average goals for country
pairRdd_goals_country = pairRdd.map(lambda x: (x[1][2], (int(x[1][1]), 1)))
pairRdd_goals_country = pairRdd_goals_country.reduceByKey(lambda x,y: (x[0]+y[0], x[1]+y[1])).mapValues(lambda x: x[0]/x[1])
pairRdd_goals_country.collect()


[('Germany', 5.153846153846154),
 ('Scotland', 2.269230769230769),
 ('Hungary', 2.3076923076923075),
 ('Switzerland', 4.8076923076923075),
 ('Spain', 3.6538461538461537),
 ('Croatia', 5.653846153846154),
 ('Albania', 1.5384615384615385),
 ('Italy', 2.269230769230769),
 ('Slovenia', 3.0),
 ('England', 3.8461538461538463),
 ('Denmark', 5.1923076923076925),
 ('Serbia', 4.961538461538462),
 ('Poland', 5.703703703703703),
 ('Netherlands', 5.423076923076923),
 ('Austria', 3.576923076923077),
 ('France', 7.68),
 ('Belgium', 7.08),
 ('Slovakia', 2.576923076923077),
 ('Romania', 2.423076923076923),
 ('Ukraine', 4.384615384615385),
 ('Turkiye', 2.730769230769231),
 ('Georgia', 2.4615384615384617),
 ('Portugal', 8.653846153846153),
 ('Czech republic', 2.5)]